# Score a GerryChain run

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

`PlanEvaluator` can score partitions after a run or expose the same metrics as GerryChain
updaters during the run. This example uses a short ReCom chain on a small grid so the two
workflows can be compared directly.

In [ ]:
import networkx as nx
import pandas as pd
from gerrychain import MarkovChain, Partition
from gerrychain.accept import always_accept
from gerrychain.constraints import contiguous
from gerrychain.proposals import build_recom_proposal_fn

import gerrytools.scoring as gs

## Build a small chain

The grid has 36 nodes and four starting districts. Population and two vote columns live on
the graph in the same form they would take on a state dual graph.

In [ ]:
grid = nx.grid_2d_graph(6, 6)
for row, column in grid:
    grid.nodes[(row, column)].update(
        population=1,
        district=(row * 6 + column) // 9,
        democratic=30 + 4 * column + row,
        republican=50 - 2 * column + row,
    )

Register the metrics once. `Tally` returns district populations, `Seats` returns one value
for the plan, and `CutEdges` returns a plan-level count. The explicit `cut_edge_count` name
keeps GerryTools' score distinct from GerryChain's built-in `cut_edges` updater.

In [ ]:
evaluator = gs.PlanEvaluator(grid).add_metrics(
    gs.Tally("population"),
    gs.Seats("democratic", "republican", name="democratic_seats"),
    gs.CutEdges(name="cut_edge_count"),
)

## Use the metrics as updaters

`to_updaters()` returns a mapping accepted directly by `Partition`. The first registered
metric requested on a partition computes all three scores; GerryChain then caches them on
that partition. Here the population result also supplies the updater required by ReCom.

In [ ]:
initial = Partition(grid, "district", updaters=evaluator.to_updaters())
chain = MarkovChain(
    build_recom_proposal_fn("population", pop_target=9, epsilon=0.15),
    constraints=[contiguous],
    acceptance_fn=always_accept,
    initial_partition=initial,
    total_steps=4,
    rng=0,
)

The updater values are ordinary pandas objects or scalars. Access them by name while the
chain is running, alongside any other GerryChain updater.

In [ ]:
plans = list(chain)
pd.DataFrame(
    {
        "Democratic seats": [plan["democratic_seats"] for plan in plans],
        "Cut edges": [plan["cut_edge_count"] for plan in plans],
    },
)

## Evaluate selected plans together

The evaluator also accepts existing partitions. `evaluate_many()` is useful when only a few
plans need to be compared, whether they came from this chain, another run, or saved
assignments. `sample_ids` become the row labels in every returned table.

In [ ]:
selected = evaluator.evaluate_many(
    [plans[0], plans[-1]],
    sample_ids=["initial", "last"],
)
selected["population"]

In [ ]:
pd.DataFrame(
    {
        "Democratic seats": selected["democratic_seats"],
        "Cut edges": selected["cut_edge_count"],
    }
)

For a long recorded run, use the [BENDL scoring tutorial](bendl.ipynb) to select individual
assignments or stream the full ensemble without loading every plan into memory. The
[scoring overview](index.md) lists the available metric families and result shapes.